# 10 — First-half experiment and error audit

This notebook evaluates the current visible-ball settling candidate against the locally available first-half reference. The reference is used only after the prediction is generated: it is evaluation, not detector training or rule fitting.

The audit reports frame errors, contiguous error intervals, and the scenario/transition context emitted by the pipeline.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'evaluation'))

gt_candidates = [
    ROOT / 'data' / 'first_half_gt.csv',
    ROOT.parent / 'barca-hack-repo-staging' / 'data' / 'first_half_gt.csv',
    ROOT / 'data' / 'sample' / 'first_half_gt.csv',
]
gt_path = next((path for path in gt_candidates if path.exists()), None)
prediction_path = ROOT / 'evaluation' / 'results' / 'diagnostic_visible_ball_low_motion_dead_ball.csv'
report_dir = ROOT / 'evaluation' / 'reports' / 'first_half'
assert gt_path is not None, f'first-half reference not found: {gt_candidates}'
assert prediction_path.exists(), f'candidate missing: {prediction_path}'
print('ground truth:', gt_path)
print('prediction:', prediction_path)

In [ ]:
import possession_metrics as pm

gt = pm.read_label_csv(gt_path)
pred = pm.read_label_csv(prediction_path)
assert gt['frame'].equals(pred['frame']), 'ground truth and prediction frames are not aligned'
print('GT validation:', pm.validate_submission(gt, len(pred)) or 'ok')
print('prediction validation:', pm.validate_submission(pred, len(gt)) or 'ok')
scores = pm.evaluate(pm.label_codes(gt), pm.label_codes(pred), bootstrap_n=0)
score_summary = {
    'macro_f1': scores['macro_f1'],
    'spell_f1_mean': scores['spell_f1_mean'],
    'n_graded': scores['n_graded'],
    'n_gt_spells': scores['n_gt_spells'],
    **{f'f1_{label}': values['f1'] for label, values in scores['per_class'].items()},
    **{f'precision_{label}': values['precision'] for label, values in scores['per_class'].items()},
    **{f'recall_{label}': values['recall'] for label, values in scores['per_class'].items()},
}
(report_dir / 'score_summary.json').write_text(json.dumps(score_summary, indent=2))
display(pd.Series(score_summary, name='first_half'))

In [ ]:
scoreboard = pd.read_csv(report_dir / 'scoreboard.csv')
display(scoreboard.sort_values(['macro_f1', 'spell_f1_mean'], ascending=False).head(20))

## Evaluation-only error analysis

The frame-level scenario confidence report is joined after scoring. It is diagnostic context: it does not alter the candidate labels. Episode-level scenario summaries are intentionally not joined because they do not have one row per frame.

In [ ]:
from possession_lab.mislabel_audit import build_mislabel_audit

audit = build_mislabel_audit(
    gt_path,
    prediction_path,
    report_dir,
    scenario_path=ROOT / 'evaluation' / 'reports' / 'scenario_confidence_scores.csv',
)
print('graded frames:', int(audit['audit']['graded'].sum()))
print('mislabelled frames:', len(audit['errors']))
print('contiguous error runs:', len(audit['runs']))

display(audit['summary'])
display(audit['runs'].sort_values('duration_frames', ascending=False).head(25))
display(audit['context_summary'].head(40))